# Stage 06 — Retrieval metrics and the baseline

**Track A (Buse) · Stage 6 of 10**

| | |
|---|---|
| **Input** | The eval set from stage 05, the retriever from stage 04 |
| **Output** | A baseline score, an MLflow run, and the numbers that fill the tactic ledger |
| **Promotes to** | `eval/metrics/retrieval_B.py` |

## This notebook is the referee

Everything shortlisted in stage 00 gets decided here. It is also where the thresholds
in `eval/thresholds_B.yaml` come from: you cannot pick a threshold before you know
what your own baseline scores.

## The four metrics and what each is for

| Metric | Question it answers | Why we track it |
|---|---|---|
| **nDCG@5** | Are the best chunks at the top of what the tools return? | The headline. Grade-aware, position-aware, and it is what a reranker moves. |
| **MRR** | How far down is the first good hit? | Sensitive to the single worst failure mode, a good answer buried at rank 5. |
| **recall@20** | Did the candidate set contain the answer at all? | The ceiling. If this is low, reranking is pointless and the fix is upstream in chunking or fusion. |
| **citation precision** | Do returned page ranges actually support the claim? | The only metric that connects retrieval to what the user sees. It is what Sude's editor agent ends up arguing with. |

Track all four. nDCG alone hides the difference between "ranking is bad" and
"search never found it", and those two have opposite fixes.

In [ ]:
from _nbsetup_B import REPO, load_cfg, resolve
import json, math, collections
from pathlib import Path
import numpy as np, pandas as pd

queries = [json.loads(l) for l in (REPO / "eval/datasets/queries_B.jsonl")
           .read_text(encoding="utf-8").splitlines() if l.strip()]
qrels = collections.defaultdict(dict)
for l in (REPO / "eval/datasets/qrels_B.jsonl").read_text(encoding="utf-8").splitlines():
    if l.strip():
        r = json.loads(l); qrels[r["query_id"]][r["chunk_id"]] = r["grade"]
print(len(queries), "queries,", sum(len(v) for v in qrels.values()), "labels")

In [ ]:
def dcg(grades):
    return sum((2 ** g - 1) / math.log2(i + 2) for i, g in enumerate(grades))

def ndcg_at_k(ranked_ids, rel, k=5):
    gains = [rel.get(cid, 0) for cid in ranked_ids[:k]]
    ideal = sorted(rel.values(), reverse=True)[:k]
    denom = dcg(ideal)
    return dcg(gains) / denom if denom else 0.0

def mrr(ranked_ids, rel, min_grade=2):
    for i, cid in enumerate(ranked_ids, start=1):
        if rel.get(cid, 0) >= min_grade:
            return 1.0 / i
    return 0.0

def recall_at_k(ranked_ids, rel, k=20, min_grade=2):
    good = {cid for cid, g in rel.items() if g >= min_grade}
    if not good:
        return float("nan")
    return len(good & set(ranked_ids[:k])) / len(good)

def citation_precision(ranked_ids, rel, k=5, min_grade=2):
    # Of the chunks we would actually cite, how many genuinely support an answer.
    top = ranked_ids[:k]
    return sum(1 for cid in top if rel.get(cid, 0) >= min_grade) / max(len(top), 1)

In [ ]:
from research_assistant.retrieval.service_B import search  # promoted from notebook 04

def evaluate(search_fn, k=5, candidate_k=20, label=""):
    rows = []
    for q in queries:
        rel = qrels[q["query_id"]]
        ranked = [c["chunk_id"] for c in search_fn(q["query"], candidate_k=candidate_k)]
        rows.append(dict(
            query_id=q["query_id"], shape=q["shape"],
            ndcg5=ndcg_at_k(ranked, rel, k),
            mrr=mrr(ranked, rel),
            recall20=recall_at_k(ranked, rel, candidate_k),
            cite_prec=citation_precision(ranked, rel, k),
        ))
    df = pd.DataFrame(rows)
    summary = df[["ndcg5", "mrr", "recall20", "cite_prec"]].mean().to_dict()
    print(label, {k2: round(v, 3) for k2, v in summary.items()})
    return df, summary

df_base, base = evaluate(search, label="baseline hybrid + naive rank")

### Error analysis, which matters more than the average

The mean tells you nothing actionable. Split the failures into the two kinds that
have different fixes.

- **Recall failures** (`recall20 == 0`): the answer never entered the candidate set.
  Fix upstream: chunking, fusion weights, or the query itself is out of scope.
- **Ranking failures** (`recall20 > 0` but `ndcg5` low): the answer was there and the
  ranking buried it. This is exactly what stage 08 is for.

The ratio between these two tells you whether the DPO reranker is worth building for
your corpus, or whether your time is better spent back in stage 02.

In [ ]:
recall_fail = df_base[df_base["recall20"] == 0]
rank_fail = df_base[(df_base["recall20"] > 0) & (df_base["ndcg5"] < 0.4)]
print(f"recall failures: {len(recall_fail)}  (fix in stages 02-04)")
print(f"ranking failures: {len(rank_fail)}  (fix in stage 08)")
print("\nworst queries:")
print(df_base.sort_values("ndcg5").head(8)[["query_id", "shape", "ndcg5", "recall20"]])
print("\nby question shape:")
print(df_base.groupby("shape")[["ndcg5", "recall20"]].mean())

In [ ]:
import mlflow
rcfg = load_cfg("reranker")
mlflow.set_tracking_uri(rcfg["mlflow"]["tracking_uri"])
mlflow.set_experiment("retrieval_baseline_B")

with mlflow.start_run(run_name="hybrid_rrf_baseline"):
    icfg = load_cfg("ingestion"); vcfg = load_cfg("retrieval")
    mlflow.log_params({
        "embed_model": icfg["embed"]["model"],
        "chunk_strategy": icfg["chunk"]["strategy"],
        "target_tokens": icfg["chunk"]["target_tokens"],
        "fusion": vcfg["hybrid"]["fusion"],
        "candidate_k": vcfg["hybrid"]["candidate_k"],
        "top_k": vcfg["rerank"]["top_k"],
    })
    mlflow.log_metrics({k: float(v) for k, v in base.items()})
print("logged. compare runs with:  mlflow ui --backend-store-uri ./mlruns")

### Setting the gate thresholds

Now, and not before, open `eval/thresholds_B.yaml` and set the minimums.

The rule: **set each minimum slightly below your current baseline**, not at some
round number you like. A threshold above your own baseline blocks every pull request
including yours. A threshold far below it never fires. Then set
`regression_tolerance` to roughly the run-to-run noise you observe by re-running this
notebook twice with different seeds.

Tell Sude the numbers and the reasoning, because her gate is the thing that enforces them.

## Exit checks

- [ ] Baseline numbers for all four metrics are recorded in MLflow.
- [ ] Thresholds are set from the baseline, and you can defend each one in a sentence.
- [ ] Failures are split into recall versus ranking, with counts.
- [ ] Every shortlisted tactic from stage 00 now has a measured delta in the ledger,
      including the ones you rejected. Rejections with numbers are the strongest part
      of a methods write-up.

## Promote to `src/`

The four metric functions and `evaluate` go to `eval/metrics/retrieval_B.py`. Sude's
gate runner imports them directly, so keep their signatures stable.